# Part II — Solution Geometry and Implicit Bias
**Author:** Zhiliang Wang (zw3162) 
**Course:** COMS 6699 Final Project

Measures four geometry quantities after training:
1. L2 distance from initialization  
2. Average gradient norm per epoch  
3. Hessian sharpness λ_max (power iteration)  
4. Hessian trace tr(H) (Hutchinson estimator)  
5. Inter-seed parameter dispersion

**Expected runtime on T4 GPU: ~25 min for all 20 runs.**

In [ ]:
# ── 0. Check GPU ─────────────────────────────────────────────────────────────
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

In [ ]:
# ── 1. Mount Google Drive (optional — for saving results) ────────────────────
# Uncomment if you want results saved to Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# RESULTS_DIR = '/content/drive/MyDrive/6699_part2/results_part2'
# FIGURES_DIR = '/content/drive/MyDrive/6699_part2/figures_part2'

RESULTS_DIR = './results_part2'
FIGURES_DIR = './figures_part2'
import os
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
# ── 2. Imports & global config ───────────────────────────────────────────────
import json, time, itertools
import numpy as np
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Hyperparameters (Table 1 in the paper) ───────────────────────────────────
SEEDS      = [42, 123, 456, 789, 2024]
EPOCHS     = 30
BATCH_SIZE = 128
SGD_LR     = 0.01;  SGD_MOM = 0.9
ADAM_LR    = 0.001; ADAM_B1 = 0.9; ADAM_B2 = 0.999

# Hessian estimation settings
SHARP_STEPS  = 100   # power iteration steps
HESS_PROBE   = 30    # Hutchinson probe vectors
HESS_SAMPLES = 512   # mini-batch size for Hessian estimation

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

In [ ]:
# ── 3. Models ────────────────────────────────────────────────────────────────

class MLP(nn.Module):
    """Linear(784,256)->BN->ReLU -> Linear(256,128)->BN->ReLU -> Linear(128,10)"""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.net(x.view(x.size(0), -1))


class SmallCNN(nn.Module):
    """Two conv blocks -> Linear(1568,128)->ReLU -> Linear(128,10)"""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(1568, 128), nn.ReLU(), nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.classifier(self.features(x).view(x.size(0), -1))


ARCHITECTURES = {'MLP': MLP, 'SmallCNN': SmallCNN}

def get_flat_params(model):
    return torch.cat([p.detach().flatten() for p in model.parameters()])

# Quick shape check
x = torch.randn(4, 1, 28, 28)
print('MLP output:', MLP()(x).shape, '  params:', get_flat_params(MLP()).numel())
print('CNN output:', SmallCNN()(x).shape, '  params:', get_flat_params(SmallCNN()).numel())

In [ ]:
# ── 4. Data loading (matched-seed design) ────────────────────────────────────

FMNIST_MEAN, FMNIST_STD = 0.2860, 0.3530
_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((FMNIST_MEAN,), (FMNIST_STD,)),
])

def get_loaders(seed):
    """Return (train_loader, test_loader) with deterministic shuffle seeded by `seed`."""
    train_set = datasets.FashionMNIST('./data', train=True,  download=True, transform=_transform)
    test_set  = datasets.FashionMNIST('./data', train=False, download=True, transform=_transform)
    g = torch.Generator()
    g.manual_seed(seed)
    train_loader = torch.utils.data.DataLoader(
        train_set, batch_size=BATCH_SIZE, shuffle=True,
        generator=g, num_workers=2, pin_memory=True)
    test_loader = torch.utils.data.DataLoader(
        test_set, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, test_loader

print('Downloading Fashion-MNIST...')
_, _ = get_loaders(42)
print('Done.')

In [ ]:
# ── 5. Geometry: Hessian-vector product (Pearlmutter trick) ──────────────────

def hv_product(loss, params, v_flat):
    """
    Compute H*v without forming H.
    Uses the identity: Hv = ∇_θ [ (∇_θ L)^T v ]
    """
    grads = torch.autograd.grad(loss, params, create_graph=True, retain_graph=True)
    grad_flat = torch.cat([g.flatten() for g in grads])
    grad_v = (grad_flat * v_flat.detach()).sum()
    hv = torch.autograd.grad(grad_v, params, retain_graph=False)
    return torch.cat([h.detach().flatten() for h in hv])


def sharpness_power_iter(model, loader, n_steps=100, tol=1e-4, max_samp=512):
    """
    Estimate λ_max(H) by power iteration.
    Iterates:  v_{k+1} = H v_k / ||H v_k||,  eigenvalue ← v_k^T H v_k.
    """
    model.eval()
    criterion = nn.CrossEntropyLoss()
    params = [p for p in model.parameters() if p.requires_grad]
    d = sum(p.numel() for p in params)

    xs, ys = [], []
    for xb, yb in loader:
        xs.append(xb); ys.append(yb)
        if sum(x.size(0) for x in xs) >= max_samp: break
    x_b = torch.cat(xs)[:max_samp].to(DEVICE)
    y_b = torch.cat(ys)[:max_samp].to(DEVICE)

    v = torch.randn(d, device=DEVICE)
    v /= v.norm()
    lam = 0.0

    for _ in range(n_steps):
        model.zero_grad()
        loss = criterion(model(x_b), y_b)
        hv   = hv_product(loss, params, v)
        new_lam = (v * hv).sum().item()
        v = hv / (hv.norm() + 1e-12)
        if abs(new_lam - lam) < tol:
            lam = new_lam; break
        lam = new_lam

    return float(lam)


def hessian_trace_hutchinson(model, loader, n_probe=30, max_samp=512):
    """
    Estimate tr(H) via Hutchinson's estimator.
    tr(H) = E[z^T H z]  for z ~ Rademacher{-1,+1}^d.
    Average over n_probe independent draws.
    """
    model.eval()
    criterion = nn.CrossEntropyLoss()
    params = [p for p in model.parameters() if p.requires_grad]
    d = sum(p.numel() for p in params)

    xs, ys = [], []
    for xb, yb in loader:
        xs.append(xb); ys.append(yb)
        if sum(x.size(0) for x in xs) >= max_samp: break
    x_b = torch.cat(xs)[:max_samp].to(DEVICE)
    y_b = torch.cat(ys)[:max_samp].to(DEVICE)

    estimates = []
    for _ in range(n_probe):
        z = torch.randint(0, 2, (d,), device=DEVICE).float() * 2 - 1
        model.zero_grad()
        loss = criterion(model(x_b), y_b)
        hz = hv_product(loss, params, z)
        estimates.append((z * hz).sum().item())

    return float(np.mean(estimates))


print('Geometry functions defined.')

In [ ]:
# ── 6. Training loop ─────────────────────────────────────────────────────────

def set_seed(seed):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        correct += (model(xb).argmax(1) == yb).sum().item()
        total   += yb.size(0)
    return correct / total


def train_one_run(arch_name, opt_name, seed):
    """
    Train one (arch, optimizer, seed) run for EPOCHS epochs.
    Returns a dict with all Part II metrics.
    """
    set_seed(seed)

    model = ARCHITECTURES[arch_name]().to(DEVICE)
    init_flat = get_flat_params(model).clone()

    if opt_name == 'SGD':
        optimizer = torch.optim.SGD(model.parameters(), lr=SGD_LR,
                                    momentum=SGD_MOM, weight_decay=0)
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=ADAM_LR,
                                     betas=(ADAM_B1, ADAM_B2), weight_decay=0)

    criterion = nn.CrossEntropyLoss()
    train_loader, test_loader = get_loaders(seed)

    param_distances, grad_norms = [], []
    train_acc_curve, test_acc_curve = [], []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        ep_gnorms, ep_loss, n_b = [], 0.0, 0

        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()

            gnorm = torch.cat([
                p.grad.detach().flatten()
                for p in model.parameters() if p.grad is not None
            ]).norm().item()
            ep_gnorms.append(gnorm)
            ep_loss += loss.item(); n_b += 1
            optimizer.step()

        flat_now = get_flat_params(model)
        dist = (flat_now - init_flat).norm().item()
        param_distances.append(dist)
        grad_norms.append(float(np.mean(ep_gnorms)))

        tr_acc = evaluate(model, train_loader)
        te_acc = evaluate(model, test_loader)
        train_acc_curve.append(tr_acc)
        test_acc_curve.append(te_acc)

        if epoch % 5 == 0 or epoch == 1:
            print(f'  [{arch_name}/{opt_name}/s={seed}] '
                  f'ep{epoch:2d}/{EPOCHS}  loss={ep_loss/n_b:.4f}  '
                  f'dist={dist:.2f}  gnorm={grad_norms[-1]:.4f}  '
                  f'tr={tr_acc:.4f}  te={te_acc:.4f}', flush=True)

    # ── Hessian metrics at final epoch ────────────────────────────────────────
    print(f'  Computing λ_max...', flush=True)
    sharpness = sharpness_power_iter(
        model, train_loader, n_steps=SHARP_STEPS, max_samp=HESS_SAMPLES)

    print(f'  Computing tr(H)...', flush=True)
    trace = hessian_trace_hutchinson(
        model, train_loader, n_probe=HESS_PROBE, max_samp=HESS_SAMPLES)

    print(f'  λ_max={sharpness:.2f}  tr(H)={trace:.2f}', flush=True)

    return {
        'arch': arch_name, 'optimizer': opt_name, 'seed': seed,
        'param_distances':  param_distances,
        'grad_norms':       grad_norms,
        'train_acc_curve':  train_acc_curve,
        'test_acc_curve':   test_acc_curve,
        'final_sharpness':  float(sharpness),
        'final_trace':      float(trace),
        '_final_flat':      get_flat_params(model).cpu().numpy(),
    }

print('Training functions defined.')

In [ ]:
# ── 7. Run all experiments ───────────────────────────────────────────────────
# Total: 2 archs × 2 opts × 5 seeds = 20 runs
# Expected time on T4: ~25 minutes

t_start = time.time()
run_order = [
    (arch, opt, seed)
    for arch in ['MLP', 'SmallCNN']
    for opt  in ['SGD', 'Adam']
    for seed in SEEDS
]

for i, (arch, opt, seed) in enumerate(run_order):
    out_json = f'{RESULTS_DIR}/{arch}_{opt}_seed{seed}.json'
    out_npy  = f'{RESULTS_DIR}/{arch}_{opt}_seed{seed}_params.npy'

    if os.path.exists(out_json):
        print(f'[{i+1}/{len(run_order)}] Skip (exists): {out_json}', flush=True)
        continue

    print(f'\n{"="*60}', flush=True)
    print(f'[{i+1}/{len(run_order)}] {arch} | {opt} | seed={seed}', flush=True)
    print(f'{"="*60}', flush=True)
    t0 = time.time()

    result = train_one_run(arch, opt, seed)

    np.save(out_npy, result.pop('_final_flat').astype(np.float32))
    with open(out_json, 'w') as f:
        json.dump(result, f, indent=2)

    print(f'  Saved in {time.time()-t0:.0f}s → {out_json}', flush=True)

print(f'\nAll runs done in {(time.time()-t_start)/60:.1f} min.')

In [ ]:
# ── 8. Helper: load results ──────────────────────────────────────────────────

def load_run(arch, opt, seed):
    with open(f'{RESULTS_DIR}/{arch}_{opt}_seed{seed}.json') as f:
        return json.load(f)

def collect_metric(arch, opt, key):
    """Shape: (n_seeds, n_epochs) for curve keys, (n_seeds,) for scalars."""
    return np.array([load_run(arch, opt, s)[key] for s in SEEDS])

EPOCH_AXIS = list(range(1, EPOCHS + 1))
SGD_COLOR  = '#2166ac'
ADAM_COLOR = '#d6604d'

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
})

def plot_band(ax, data, color, label):
    m, s = data.mean(0), data.std(0)
    ax.plot(EPOCH_AXIS, m, color=color, label=label, linewidth=1.8)
    ax.fill_between(EPOCH_AXIS, m-s, m+s, color=color, alpha=0.15)

print('Result loading helpers defined.')

In [ ]:
# ── 9. Figure 4: L2 distance from initialization ─────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
labels = ['(a) MLP', '(b) SmallCNN']

for ax, arch, lab in zip(axes, ['MLP', 'SmallCNN'], labels):
    plot_band(ax, collect_metric(arch, 'SGD',  'param_distances'), SGD_COLOR,  'SGD')
    plot_band(ax, collect_metric(arch, 'Adam', 'param_distances'), ADAM_COLOR, 'Adam')
    ax.set_xlabel('Epoch'); ax.set_ylabel(r'$\|\theta_t - \theta_0\|_2$')
    ax.set_title(lab); ax.legend()

fig.suptitle('L2 Distance from Initialization', fontsize=12, y=1.01)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/fig4_param_distance.pdf', bbox_inches='tight')
fig.savefig(f'{FIGURES_DIR}/fig4_param_distance.png', bbox_inches='tight', dpi=150)
plt.show(); print('Figure 4 saved.')

In [ ]:
# ── 10. Figure 5: Gradient norm per epoch ────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

for ax, arch, lab in zip(axes, ['MLP', 'SmallCNN'], ['(a) MLP', '(b) SmallCNN']):
    plot_band(ax, collect_metric(arch, 'SGD',  'grad_norms'), SGD_COLOR,  'SGD')
    plot_band(ax, collect_metric(arch, 'Adam', 'grad_norms'), ADAM_COLOR, 'Adam')
    ax.set_xlabel('Epoch'); ax.set_ylabel(r'Average $\|g_t\|_2$')
    ax.set_title(lab); ax.legend()

fig.suptitle('Average Gradient Norm per Epoch', fontsize=12, y=1.01)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/fig5_gradient_norm.pdf', bbox_inches='tight')
fig.savefig(f'{FIGURES_DIR}/fig5_gradient_norm.png', bbox_inches='tight', dpi=150)
plt.show(); print('Figure 5 saved.')

In [ ]:
# ── 11. Figure 6: Sharpness and Hessian trace bar charts ─────────────────────

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
metrics = [('final_sharpness', r'Sharpness $\lambda_{\max}(H)$'),
           ('final_trace',     r'Hessian Trace $\mathrm{tr}(H)$')]

for ax, (key, ylabel) in zip(axes, metrics):
    x = np.array([0, 1.4])
    w = 0.45
    for i, (arch, xc) in enumerate(zip(['MLP', 'SmallCNN'], x)):
        sgd_v  = collect_metric(arch, 'SGD',  key)
        adam_v = collect_metric(arch, 'Adam', key)
        kw = dict(capsize=4, linewidth=1.2)
        ax.bar(xc-w/2, sgd_v.mean(),  w, yerr=sgd_v.std(),
               color=SGD_COLOR,  label='SGD'  if i==0 else '', **kw)
        ax.bar(xc+w/2, adam_v.mean(), w, yerr=adam_v.std(),
               color=ADAM_COLOR, label='Adam' if i==0 else '', **kw)
    ax.set_xticks(x); ax.set_xticklabels(['MLP', 'SmallCNN'])
    ax.set_ylabel(ylabel); ax.legend()

fig.suptitle('Hessian-Based Curvature Proxies at Final Epoch', fontsize=12, y=1.01)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/fig6_sharpness_trace.pdf', bbox_inches='tight')
fig.savefig(f'{FIGURES_DIR}/fig6_sharpness_trace.png', bbox_inches='tight', dpi=150)
plt.show(); print('Figure 6 saved.')

In [ ]:
# ── 12. Figure 7: Inter-seed parameter dispersion ────────────────────────────

def pairwise_dists(arch, opt):
    vecs = [torch.tensor(np.load(f'{RESULTS_DIR}/{arch}_{opt}_seed{s}_params.npy'))
            for s in SEEDS]
    return [(vecs[i]-vecs[j]).norm().item()
            for i, j in itertools.combinations(range(len(vecs)), 2)]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

for ax, arch, lab in zip(axes, ['MLP', 'SmallCNN'], ['(a) MLP', '(b) SmallCNN']):
    sgd_pw  = pairwise_dists(arch, 'SGD')
    adam_pw = pairwise_dists(arch, 'Adam')

    bp = ax.boxplot([sgd_pw, adam_pw], labels=['SGD','Adam'],
                    patch_artist=True, widths=0.4,
                    medianprops={'color':'black','linewidth':2})
    for box, color in zip(bp['boxes'], [SGD_COLOR, ADAM_COLOR]):
        box.set_facecolor(color + '99'); box.set_edgecolor(color)

    rng = np.random.default_rng(0)
    for k, (vals, color) in enumerate([(sgd_pw, SGD_COLOR), (adam_pw, ADAM_COLOR)], 1):
        jitter = rng.uniform(-0.07, 0.07, len(vals))
        ax.scatter(np.full(len(vals), k) + jitter, vals,
                   color=color, s=22, alpha=0.8, zorder=3)

    ax.set_ylabel(r'Pairwise $\|\theta^*_i - \theta^*_j\|_2$')
    ax.set_title(lab)

fig.suptitle('Inter-Seed Parameter Dispersion', fontsize=12, y=1.01)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/fig7_inter_seed.pdf', bbox_inches='tight')
fig.savefig(f'{FIGURES_DIR}/fig7_inter_seed.png', bbox_inches='tight', dpi=150)
plt.show(); print('Figure 7 saved.')

In [ ]:
# ── 13. Table 3: Final scalar metrics ────────────────────────────────────────

print('\n=== Table 3: Final Scalar Metrics ===')
print(f'{"Model":<10}{"Opt":<6}{"Test acc":>14}{"Gap":>14}{"Sharpness":>18}{"Trace":>18}{"Param dist":>14}')
print('-'*96)

for arch in ['MLP', 'SmallCNN']:
    for opt in ['SGD', 'Adam']:
        te  = collect_metric(arch, opt, 'test_acc_curve')[:, -1]
        tr  = collect_metric(arch, opt, 'train_acc_curve')[:, -1]
        gap = tr - te
        sh  = collect_metric(arch, opt, 'final_sharpness')
        tr_ = collect_metric(arch, opt, 'final_trace')
        di  = collect_metric(arch, opt, 'param_distances')[:, -1]

        def fmt(arr): return f'{arr.mean():.4f}±{arr.std():.4f}'
        def fmt2(arr): return f'{arr.mean():.2f}±{arr.std():.2f}'

        print(f'{arch:<10}{opt:<6}{fmt(te):>14}{fmt(gap):>14}{fmt2(sh):>18}{fmt2(tr_):>18}{fmt2(di):>14}')

In [ ]:
# ── 14. Table 4: Inter-seed dispersion ───────────────────────────────────────

print('\n=== Table 4: Inter-Seed Parameter Dispersion ===')
print(f'{"Architecture":<14}{"SGD mean pairwise dist":>24}{"Adam mean pairwise dist":>24}')
print('-'*64)

for arch in ['MLP', 'SmallCNN']:
    sgd_d  = np.mean(pairwise_dists(arch, 'SGD'))
    adam_d = np.mean(pairwise_dists(arch, 'Adam'))
    print(f'{arch:<14}{sgd_d:>24.2f}{adam_d:>24.2f}')

In [ ]:
# ── 15. Download results ZIP (for uploading to the paper repo) ───────────────
import shutil

shutil.make_archive('part2_results', 'zip', '.', 'results_part2')
shutil.make_archive('part2_figures', 'zip', '.', 'figures_part2')

from google.colab import files
files.download('part2_results.zip')
files.download('part2_figures.zip')